## Prerequisites
1. Get PageIndex API key: https://dash.pageindex.ai/api-keys
2. Get Gemini API key: https://aistudio.google.com/app/apikey
3. Create `.env` file with both keys

In [1]:
# Install dependencies (run once)
!pip install -U pageindex python-dotenv google-generativeai ipykernel




Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = ""
GEMINI_API_KEY    ="" #THIS IS NOT A GOOD WAY TO FETCH API KEYS. USE .env, AM REVOKE THAT KEYS SO DON'T WORRY

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Gemini key loaded:   ", "✅" if GEMINI_API_KEY    else "❌ Missing!")

if not (PAGEINDEX_API_KEY and GEMINI_API_KEY):
    raise ValueError("Missing API keys. Create .env file with both keys.")

PageIndex key loaded: ✅
Gemini key loaded:    ✅


In [3]:
from pageindex import PageIndexClient
import google.generativeai as genai

# Initialize clients
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

print("✅ PageIndex client ready")
print("✅ Gemini client ready")

/home/shivas/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ PageIndex client ready
✅ Gemini client ready


/home/shivas/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_5188/1574743373.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Upload PDF & Build Tree Index

In [6]:
PDF_PATH = "RAG.pdf"  # Change to your PDF path

print(f"📤 Uploading: {PDF_PATH}")
try:
    result = pi_client.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print("✅ Uploaded!")
    print(f"📋 Document ID: {doc_id}")
except Exception as e:
    print(f"❌ Error uploading: {e}")
    print("\n⚠️  TROUBLESHOOTING:")
    print("1. Check that PAGEINDEX_API_KEY is set in .env")
    print("2. Get key from: https://dash.pageindex.ai/api-keys")
    print("3. Verify PDF file path is correct")

📤 Uploading: RAG.pdf
✅ Uploaded!
📋 Document ID: pi-cmo8hgdnq05iy01pe12k3nj2k


In [7]:
# Wait for tree to finish building
print("\n⏳ Building tree index...")
max_wait = 10  # Try for 10 attempts (50 seconds)
attempt = 0

while attempt < max_wait:
    try:
        status_result = pi_client.get_document(doc_id)
        status = status_result.get("status")
        print(f"   Status: {status}")

        if status == "completed":
            print("\n✅ Tree index ready!")
            break
        elif status == "failed":
            print("\n❌ Processing failed.")
            break

        time.sleep(5)
        attempt += 1
    except Exception as e:
        print(f"Error checking status: {e}")
        break


⏳ Building tree index...
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


## Inspect the Tree Structure

In [8]:
tree_result    = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}\n")
print("🌲 First node preview:")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2)[:1000])

📊 Top-level sections: 1

🌲 First node preview:
{
  "title": "11 Information Retrieval and Retrieval-Augmented Generation",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This text discusses information retrieval and retrieval-augmented generation (RAG). It begins by tracing the history of question answering systems and the role of large language models (LLMs) in fulfilling information needs, noting the overlap with web search. The text then highlights the limitations of simply prompting LLMs for factual answers, such as hallucinations, lack of calibration, inability to access proprietary data, and static knowledge. RAG is presented as a solution, where external documents are retrieved and used by LLMs to generate grounded and contextually supported answers. The chapter will cover information retrieval techniques, the RAG paradigm, and relevant datasets for LLM training and evaluation.",
  "text": "# 11 Information Retrieval and Retrieval-Augmented Generation\n\nOn two occ

In [9]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Document Structure\n")
print_tree(pageindex_tree)

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

print(f"\n🔢 Total nodes: {count_nodes(pageindex_tree)}")

📚 Document Structure

[0000] 11 Information Retrieval and Retrieval-Augmented Generation  (p.1)
  └─ [0001] 11.1 Information Retrieval  (p.3)
    └─ [0002] 11.1.1 Representing documents as vectors  (p.4)
    └─ [0003] 11.1.2 Term weighting: tfidf and BM25  (p.5)
    └─ [0004] 11.1.3 Document Scoring  (p.7)
    └─ [0005] 11.1.4 Efficiently finding documents: the Inverted Index  (p.9)
  └─ [0006] 11.2 Evaluation of Information-Retrieval Systems  (p.10)
    └─ [0007] ColBERT  (p.14)
  └─ [0008] 11.4 Retrieval-Augmented Generation (RAG)  (p.16)
  └─ [0009] 11.5 Datasets  (p.18)
  └─ [0010] 11.6 Evaluating Question Answering  (p.20)
  └─ [0011] 11.7 Summary  (p.20)
  └─ [0012] Historical Notes  (p.21)
  └─ [0013] Exercises  (p.23)

🔢 Total nodes: 14


## LLM Tree Search (With Gemini)

In [10]:
def llm_tree_search(query: str, tree: list) -> dict:
    """Uses Gemini to reason about which nodes contain the answer"""
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150],
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Identify which node IDs most likely contain the answer.
Think step-by-step.

Query: {query}

Document Tree:
{json.dumps(compressed, indent=2)}

Reply ONLY in this JSON format:
{{
  \"thinking\": \"<your reasoning>\",
  \"node_list\": [\"node_id1\", \"node_id2\"]
}}"""

    try:
        response = gemini_model.generate_content(prompt)
        response_text = response.text
        
        # Extract JSON if wrapped in markdown
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0]
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0]
        
        return json.loads(response_text.strip())
    except json.JSONDecodeError as e:
        print(f"⚠️  JSON parsing error: {e}")
        return {"thinking": "Error parsing response", "node_list": []}
    except Exception as e:
        print(f"⚠️  Error calling Gemini: {e}")
        return {"thinking": "Error calling LLM", "node_list": []}

## Full RAG Pipeline

In [11]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

def generate_answer(query: str, nodes: list) -> str:
    """Uses Gemini to generate a cited answer"""
    if not nodes:
        return "⚠️ No relevant sections found."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
Cite the section title and page number after every claim.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""

    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"⚠️  Error generating answer: {e}"

In [12]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    if verbose:
        print("\n" + "=" * 60)
        print(f"🔍 Query: {query}")
        print("=" * 60)

    # 1️⃣  Tree Search
    search   = llm_tree_search(query, tree)
    node_ids = search.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search.get('thinking','')[:200]}...")
        print(f"🎯 Node IDs : {node_ids}")

    # 2️⃣  Retrieve
    nodes = find_nodes_by_ids(tree, node_ids)
    if verbose:
        print(f"📄 Sections : {[n['title'] for n in nodes]}")

    # 3️⃣  Generate
    answer = generate_answer(query, nodes)
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    return answer

## Test it Out!

In [13]:
# Single query demo
_ = vectorless_rag(
    query="What are the main topics covered in this document?",
    tree=pageindex_tree,
)


🔍 Query: What are the main topics covered in this document?
⚠️  Error calling Gemini: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 48.92774549s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_

In [14]:
# Multiple questions
questions = [
    "What is the primary focus of this document?",
    "What problems does this address?",
    "What are the key findings?",
]

for q in questions:
    print(f"\nQ: {q}")
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"A: {ans[:400]}...")
    print("-" * 60)


Q: What is the primary focus of this document?
⚠️  Error calling Gemini: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 48.281661614s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "G